# Contextual Compression Retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace

# this library is used for contextual compression retriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

# this library is used to compress document through LLM
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [12]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [13]:
# Create a FAISS vector store from the documents
embedding_model = HuggingFaceEmbeddings()
vector_store = FAISS.from_documents(docs, embedding_model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 445.47it/s]


In [14]:
base_retriver = vector_store.as_retriever(search_kwargs={"k": 3})

In [15]:
# Initialize Hugging Face Endpoint and Chat Model
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 256, "temperature": 0.3, "do_sample": True},
)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 212.87it/s]


In [16]:
# Set up the compressor using an LLM

llm = ChatHuggingFace(llm=llm)
compressor = LLMChainExtractor.from_llm(llm)

In [17]:
# Create the contextual compression retriever

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriver,
    base_compressor=compressor
)

In [19]:
# Query the retriever
query = "What is Photosynthesis?"

compressed_results = compression_retriever.invoke(query)

In [20]:
# print retrieve content

for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the following question and context, extract any part of the context *AS IS* that is relevant to answer the question. If none of the context is relevant return NO_OUTPUT.

Remember, *DO NOT* edit the extracted parts of the context.

> Question: What is Photosynthesis?
> Context:
>>>
The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years.
>>>
Extracted relevant parts:<|im_end|>
<|im_start|>assistant
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the following question and context